# Course 11: Introduction to Generative AI
## GCP Machine Learning Engineer Certification

This notebook covers hands-on exercises for generative AI concepts on Google Cloud:
- Vertex AI GenAI SDK setup
- Gemini API: text generation and multimodal (text + image)
- Prompt design: zero-shot, few-shot, chain-of-thought
- Temperature and top-k/top-p sampling comparison
- Content safety filters
- Simple chatbot with conversation history

**Prerequisites**: A GCP project with Vertex AI API enabled and billing configured.

---
## 1. Environment Setup

In [ ]:
# Install the Vertex AI SDK for GenAI
!pip install -q google-cloud-aiplatform>=1.38.0 google-generativeai>=0.3.0

In [ ]:
# Authenticate (Colab)
from google.colab import auth
auth.authenticate_user()

# Set your project and location
PROJECT_ID = "your-project-id"  # <-- Replace with your GCP project ID
LOCATION = "us-central1"

import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)

print(f"Vertex AI initialized: project={PROJECT_ID}, location={LOCATION}")

In [ ]:
# Import the generative model classes
from vertexai.generative_models import GenerativeModel, Part, SafetySetting, HarmCategory, HarmBlockThreshold
import vertexai.generative_models as genai

print("Imports successful.")

---
## 2. Gemini API: Text Generation

In [ ]:
# Initialize the Gemini 1.5 Pro model
model = GenerativeModel("gemini-1.5-pro")

# Simple text generation
response = model.generate_content(
    "Explain what generative AI is in 3 sentences, suitable for a business executive."
)

print("=== Gemini Text Generation ===")
print(response.text)

In [ ]:
# Inspect the response metadata
print("Finish reason:", response.candidates[0].finish_reason)
print("Safety ratings:")
for rating in response.candidates[0].safety_ratings:
    print(f"  {rating.category.name}: {rating.probability.name}")

# Usage metadata (token counts)
print(f"\nPrompt tokens: {response.usage_metadata.prompt_token_count}")
print(f"Response tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total tokens: {response.usage_metadata.total_token_count}")

---
## 3. Multimodal Generation (Text + Image)

In [ ]:
import requests
from IPython.display import Image, display

# Download a sample image
image_url = "https://storage.googleapis.com/cloud-samples-data/ai-platform/flowers/daisy/100080576_f52e8ee070_n.jpg"
image_bytes = requests.get(image_url).content

# Display the image
display(Image(data=image_bytes, width=300))

In [ ]:
# Multimodal prompt: text + image
image_part = Part.from_data(data=image_bytes, mime_type="image/jpeg")

multimodal_response = model.generate_content(
    [
        image_part,
        "Describe this image in detail. What type of flower is it? What setting is it in?"
    ]
)

print("=== Multimodal Response ===")
print(multimodal_response.text)

---
## 4. Prompt Design: Zero-Shot, Few-Shot, Chain-of-Thought

In [ ]:
# --- ZERO-SHOT ---
# No examples provided; the model relies entirely on pre-trained knowledge

zero_shot_prompt = """Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The new Pixel phone has an amazing camera but the battery life is disappointing."

Sentiment:"""

response_zero = model.generate_content(zero_shot_prompt)
print("=== Zero-Shot ===")
print(response_zero.text)

In [ ]:
# --- FEW-SHOT ---
# Provide examples to guide the model's output format and behavior

few_shot_prompt = """Classify the sentiment of each review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "Absolutely love this product! Best purchase I've made all year."
Sentiment: POSITIVE

Review: "The product arrived broken and customer service was unhelpful."
Sentiment: NEGATIVE

Review: "It works as described. Nothing special, nothing bad."
Sentiment: NEUTRAL

Review: "The new Pixel phone has an amazing camera but the battery life is disappointing."
Sentiment:"""

response_few = model.generate_content(few_shot_prompt)
print("=== Few-Shot ===")
print(response_few.text)

In [ ]:
# --- CHAIN-OF-THOUGHT ---
# Ask the model to reason step by step before giving a final answer

cot_prompt = """A store sells apples for $1.50 each and oranges for $2.00 each.
Sarah buys 4 apples and 3 oranges. She pays with a $20 bill.
How much change does she receive?

Think step by step before giving the final answer."""

response_cot = model.generate_content(cot_prompt)
print("=== Chain-of-Thought ===")
print(response_cot.text)

### Comparison Summary

| Technique | When to Use | Pros | Cons |
|-----------|------------|------|------|
| **Zero-shot** | Simple tasks, well-defined outputs | No examples needed, fewer tokens | Less control over format |
| **Few-shot** | Format-specific outputs, classification | Consistent formatting, better accuracy | Uses more tokens |
| **Chain-of-thought** | Reasoning, math, complex logic | Better accuracy on hard problems | Longer responses, higher cost |

---
## 5. Temperature and Sampling Parameters

In [ ]:
from vertexai.generative_models import GenerationConfig

creative_prompt = "Write a one-sentence tagline for a new AI-powered coffee shop."

# Compare different temperature settings
temperatures = [0.0, 0.5, 1.0, 1.5]

print("=== Temperature Comparison ===")
print(f"Prompt: {creative_prompt}\n")

for temp in temperatures:
    config = GenerationConfig(
        temperature=temp,
        max_output_tokens=100,
    )
    response = model.generate_content(creative_prompt, generation_config=config)
    print(f"Temperature {temp}: {response.text.strip()}")

In [ ]:
# Compare top-k and top-p sampling
print("=== Top-K and Top-P Comparison ===")
print(f"Prompt: {creative_prompt}\n")

configs = {
    "top_k=1 (greedy)": GenerationConfig(temperature=0.8, top_k=1, max_output_tokens=100),
    "top_k=10 (narrow)": GenerationConfig(temperature=0.8, top_k=10, max_output_tokens=100),
    "top_k=40 (default)": GenerationConfig(temperature=0.8, top_k=40, max_output_tokens=100),
    "top_p=0.1 (focused)": GenerationConfig(temperature=0.8, top_p=0.1, max_output_tokens=100),
    "top_p=0.5 (balanced)": GenerationConfig(temperature=0.8, top_p=0.5, max_output_tokens=100),
    "top_p=0.95 (diverse)": GenerationConfig(temperature=0.8, top_p=0.95, max_output_tokens=100),
}

for name, config in configs.items():
    response = model.generate_content(creative_prompt, generation_config=config)
    print(f"{name}: {response.text.strip()}")

### Sampling Parameters Explained

- **Temperature**: Controls randomness. 0 = deterministic, >1 = more creative/random.
- **Top-K**: Limits sampling to the K most probable tokens. Lower K = more focused.
- **Top-P (nucleus sampling)**: Limits sampling to the smallest set of tokens whose cumulative probability exceeds P. Lower P = more focused.

**Exam tip**: Temperature 0 gives the most deterministic results. For factual/analytical tasks, use low temperature (0-0.3). For creative tasks, use higher temperature (0.7-1.0).

---
## 6. Content Safety Filters

In [ ]:
# Configure safety settings
# Vertex AI provides configurable safety filters for different harm categories

safety_settings = [
    SafetySetting(
        category=HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    SafetySetting(
        category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    SafetySetting(
        category=HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
        threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
    SafetySetting(
        category=HarmCategory.HARM_CATEGORY_HARASSMENT,
        threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    ),
]

print("Safety filter categories:")
print("- HARM_CATEGORY_HATE_SPEECH")
print("- HARM_CATEGORY_DANGEROUS_CONTENT")
print("- HARM_CATEGORY_SEXUALLY_EXPLICIT")
print("- HARM_CATEGORY_HARASSMENT")
print("\nThreshold levels:")
print("- BLOCK_NONE: No filtering")
print("- BLOCK_LOW_AND_ABOVE: Strictest filtering")
print("- BLOCK_MEDIUM_AND_ABOVE: Moderate filtering")
print("- BLOCK_ONLY_HIGH: Most permissive filtering")

In [ ]:
# Test with safety settings applied
safe_response = model.generate_content(
    "Write a short poem about a sunny day in the park.",
    safety_settings=safety_settings,
)

print("=== Safe Content Generation ===")
print(safe_response.text)
print("\nSafety ratings:")
for rating in safe_response.candidates[0].safety_ratings:
    print(f"  {rating.category.name}: {rating.probability.name}")

---
## 7. Simple Chatbot with Conversation History

In [ ]:
# Gemini supports multi-turn conversations natively with chat sessions

chat_model = GenerativeModel(
    "gemini-1.5-pro",
    system_instruction="You are a helpful GCP study assistant. Answer questions about Google Cloud "
                       "services concisely and accurately. If you are unsure, say so."
)

# Start a chat session
chat = chat_model.start_chat()

print("=== GCP Study Chatbot ===")
print("(Type 'quit' to exit)\n")

In [ ]:
# Turn 1
response1 = chat.send_message("What is Vertex AI Model Garden?")
print(f"User: What is Vertex AI Model Garden?")
print(f"Bot: {response1.text}\n")

In [ ]:
# Turn 2 - follow-up question (model remembers context)
response2 = chat.send_message("How does it differ from Vertex AI Studio?")
print(f"User: How does it differ from Vertex AI Studio?")
print(f"Bot: {response2.text}\n")

In [ ]:
# Turn 3 - the model should remember the full conversation
response3 = chat.send_message("Which one should I use first when starting a new GenAI project?")
print(f"User: Which one should I use first when starting a new GenAI project?")
print(f"Bot: {response3.text}\n")

In [ ]:
# Inspect conversation history
print("=== Conversation History ===")
for i, message in enumerate(chat.history):
    role = "User" if message.role == "user" else "Model"
    text_preview = message.parts[0].text[:80] + "..." if len(message.parts[0].text) > 80 else message.parts[0].text
    print(f"  [{i}] {role}: {text_preview}")

---
## Summary

In this notebook, you learned:

1. **SDK Setup**: How to initialize Vertex AI and import generative model classes
2. **Text Generation**: Basic text generation with Gemini and response inspection
3. **Multimodal**: Sending images + text to Gemini for visual understanding
4. **Prompt Design**: Zero-shot, few-shot, and chain-of-thought techniques
5. **Sampling**: How temperature, top-k, and top-p affect output diversity
6. **Safety**: Configuring content safety filters for responsible deployment
7. **Chat**: Building multi-turn conversations with conversation history

**Next**: [Course 12 - Introduction to Large Language Models](12-intro-llms.ipynb)